# Colab smoke test

Run all cells top to bottom on a **Colab T4** runtime. Every step records PASS / FAIL and the run continues,
so you get the full picture in the last cell. Takes roughly 10-15 minutes the first time (dataset download + image cache).

Before you start: upload your `.env` to `/content/.env` (needs `KAGGLE_USERNAME` and `KAGGLE_KEY`) and push the `param-tuning` branch.

In [ ]:
import contextlib, os, shutil, subprocess, sys, time, traceback
from pathlib import Path

CHECKS = {}


@contextlib.contextmanager
def check(name):
    """Record PASS/FAIL for a step and keep going, so one broken step doesn't hide the others."""
    try:
        yield
        CHECKS[name] = "PASS"
    except Exception:
        CHECKS[name] = "FAIL"
        print(traceback.format_exc())


with check("GPU available"):
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    print(f"GPU: {gpu or 'NONE'} | CPUs: {os.cpu_count()} | RAM: {ram:.1f} GB | python {sys.version.split()[0]}")
    print(f"free disk: {shutil.disk_usage('/content').free / 1e9:.0f} GB")
    assert gpu, "no GPU: Runtime > Change runtime type > T4 GPU"

In [ ]:
REPO_URL = "https://github.com/vumenira/CNN_for_multilabel_classification.git"
BRANCH = "param-tuning"
REPO = Path("/content/CNN_for_multilabel_classification")
ENV_FILE = Path("/content/.env")

with check("code cloned from the branch"):
    if not REPO.exists():
        url, token = REPO_URL, None
        if ENV_FILE.exists():  # private repo: put GITHUB_TOKEN=... into .env
            lines = ENV_FILE.read_text().splitlines()
            token = next((l.split("=", 1)[1].strip() for l in lines if l.startswith("GITHUB_TOKEN=")), None)
            if token:
                url = url.replace("https://", f"https://{token}@")
        r = subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
        assert r.returncode == 0, r.stderr.replace(token, "***") if token else r.stderr
    sys.path.insert(0, str(REPO))
    os.chdir(REPO)
    print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
    assert (REPO / "src" / "sweep.py").exists(), "the branch on GitHub lacks the new src files: commit and push it first"

In [ ]:
with check("dependencies installed"):
    packages = ["lightning", "timm", "mlflow", "albumentations>=2.0.8", "imagehash",
                "iterative-stratification", "torchmetrics", "optuna", "kaggle", "tqdm"]
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], capture_output=True, text=True)
    print(r.stderr[-1500:])
    assert r.returncode == 0
    import importlib
    for name in ["torch", "lightning", "timm", "mlflow", "albumentations", "imagehash", "iterstrat", "torchmetrics", "optuna", "cv2"]:
        module = importlib.import_module(name)
        print(f"{name:15s} {getattr(module, '__version__', '')}")
    import torch
    assert torch.cuda.is_available()
# If an import above fails with a numpy/binary error: Runtime > Restart session, then run again from the top.

In [ ]:
with check("src imports"):
    from src import colab, results, sweep
    from src.config import DATA_DIR, FOLDS_PATH, LABELS, ROOT

drive_ok = False
with check("Google Drive mounts and is writable"):
    drive_ok = colab.mount_drive()
    assert drive_ok, "Drive unavailable: results would only live on the VM"

RESULTS = (colab.DRIVE_DIR if drive_ok else Path("/content")) / "results_smoke"
RESULTS.mkdir(parents=True, exist_ok=True)
print("exports go to:", RESULTS)

with check(".env has the Kaggle keys"):
    keys = colab.load_env(ENV_FILE)
    print("keys found:", keys)
    assert "KAGGLE_USERNAME" in keys and "KAGGLE_KEY" in keys

In [ ]:
import numpy as np
import pandas as pd

with check("dataset downloaded and consistent"):
    t = time.time()
    colab.ensure_dataset(colab.DRIVE_DIR if drive_ok else None)
    print(f"dataset ready in {time.time() - t:.0f} s")
    train_csv, test_csv = pd.read_csv(DATA_DIR / "train.csv"), pd.read_csv(DATA_DIR / "test.csv")
    folds = pd.read_csv(FOLDS_PATH)
    n_train, n_test = len(os.listdir(DATA_DIR / "train")), len(os.listdir(DATA_DIR / "test"))
    print(f"train: {n_train} files / {len(train_csv)} rows | test: {n_test} files / {len(test_csv)} rows")
    assert n_train == len(train_csv) and n_test == len(test_csv)
    assert set(folds.image_id) == set(train_csv.image_id) and sorted(folds.fold.unique()) == [0, 1, 2, 3, 4]

In [ ]:
import albumentations as A
from PIL import Image
from src.data import build_cache, cache_path, split_ids

with check("image caches build (train + test)"):
    t = time.time()
    build_cache("train", 224)
    print(f"train cache: {time.time() - t:.0f} s")
    t = time.time()
    build_cache("test", 224)
    print(f"test cache: {time.time() - t:.0f} s")
    train_cache = np.load(cache_path("train", 224), mmap_mode="r")
    assert train_cache.shape == (len(train_csv), 224, 224, 3), train_cache.shape

with check("cached pixels equal the normal eval pipeline"):
    ids, diffs = split_ids("train"), []
    for row in (0, len(ids) // 2, len(ids) - 1):
        original = np.asarray(Image.open(DATA_DIR / "train" / f"{ids[row]}.jpg").convert("RGB"))
        reference = A.Resize(224, 224)(image=original)["image"]
        diffs.append(np.abs(reference.astype(int) - train_cache[row].astype(int)).mean())
    print("mean abs pixel difference:", diffs)
    assert max(diffs) < 1.0

In [ ]:
from src.cnn import CNN
from src.data import DataModule

SPEED = {}


def loader_speed(cache, n=25):
    dm = DataModule(mean=[0.5] * 3, std=[0.25] * 3, fold=0, cache=cache, num_workers=2)
    dm.setup("fit")
    batches = iter(dm.train_dataloader())
    next(batches)  # worker start-up
    t, images = time.time(), 0
    for _ in range(n):
        images += len(next(batches)[0])
    return images / (time.time() - t)


def gpu_speed(n=25, **model_kwargs):
    model = CNN(**model_kwargs).cuda()
    optimizer = torch.optim.AdamW(model.parameters())
    scaler = torch.amp.GradScaler()
    x, y = torch.randn(64, 3, 224, 224, device="cuda"), torch.rand(64, 5, device="cuda")
    for i in range(n):
        if i == 5:
            torch.cuda.synchronize()
            t = time.time()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = torch.nn.functional.binary_cross_entropy_with_logits(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
    torch.cuda.synchronize()
    return (n - 5) * 64 / (time.time() - t)


with check("throughput benchmark"):
    SPEED["loader, decode JPEGs"] = loader_speed(False)
    SPEED["loader, cached"] = loader_speed(True)
    SPEED["GPU, baseline net"] = gpu_speed()
    SPEED["GPU, largest searched net"] = gpu_speed(widths=[64, 128, 256, 512], blocks=[2, 3, 4, 2], activation="silu")
    for name, value in SPEED.items():
        print(f"{name:28s} {value:7.0f} img/s")
    train_images = int(len(train_csv) * 0.8)
    for net in ("baseline net", "largest searched net"):
        rate = min(SPEED["loader, cached"], SPEED[f"GPU, {net}"])
        print(f"estimated epoch, {net}: {train_images / rate:.0f} s (cached) vs "
              f"{train_images / min(SPEED['loader, decode JPEGs'], SPEED['GPU, ' + net]):.0f} s (JPEGs)")

## One real run, end to end (1 full epoch)

In [ ]:
EXPECTED = ["summary.json", "metrics.csv", "params.json", "curves.png", "val_probs.csv", "best.ckpt"]

with check("one full epoch trains, evaluates and exports"):
    ok = sweep.run_experiment(dict(name="smoke-ok", stage="smoke", epochs=1, max_minutes=15), RESULTS)
    assert ok["status"] == "done", ok.get("error")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in ok.items() if k in
           ("best_score", "f1_macro_05", "f1_macro_tuned", "sec_per_epoch", "num_parameters")})
    missing = [f for f in EXPECTED if not (RESULTS / "smoke-ok" / f).exists()]
    assert not missing, f"missing exports: {missing}"
    history = pd.read_csv(RESULTS / "smoke-ok" / "metrics.csv")
    for key in ("val_ap", "val_loss", "train_loss_epoch"):
        assert key in set(history.key), f"{key} was not logged"
    assert any(k.startswith("grad_norm/") for k in set(history.key)), "no gradient norms logged"
    assert (RESULTS / "mlflow.db").exists(), "MLflow database was not backed up"

with check("a finished run is skipped on re-run (resumable)"):
    t = time.time()
    again = sweep.run_experiment(dict(name="smoke-ok", stage="smoke", epochs=1, max_minutes=15), RESULTS)
    assert time.time() - t < 10 and again["run_id"] == ok["run_id"]

## Failure isolation: broken runs must not stop the queue

In [ ]:
with check("failed runs are recorded and the queue continues"):
    quick = {"limit_train_batches": 20}
    out = sweep.run_list([
        dict(name="smoke-bad-activation", stage="smoke", epochs=1, activation="nope"),
        dict(name="smoke-bad-batch", stage="smoke", epochs=1, batch_size=0),
        dict(name="smoke-after-bad", stage="smoke", epochs=1, trainer_kwargs=quick),
    ], RESULTS)
    statuses = [o["status"] for o in out]
    print(statuses)
    assert statuses == ["failed", "failed", "done"], statuses
    assert (RESULTS / "smoke-bad-batch" / "failed.json").exists()
    out = sweep.run_list([dict(name="smoke-skipped", stage="smoke")], RESULTS, deadline=time.time() - 1)
    assert out[0]["status"] == "skipped"

## Optuna study + final-run path (a few seconds per run: only 25 training batches per epoch)

In [ ]:
with check("Optuna study runs, backs up, and final runs produce test predictions"):
    study = sweep.run_optuna("smoke-optuna", n_trials=3, results_dir=RESULTS,
                             overrides=dict(epochs=3, trainer_kwargs={"limit_train_batches": 25}))
    print(study.trials_dataframe()[["number", "state", "value"]])
    assert sum(t.state.name == "COMPLETE" for t in study.trials) >= 2
    assert (RESULTS / "optuna.db").exists() and (RESULTS / "smoke-optuna_trials.csv").exists()

    finals = sweep.final_configs(study, top_k=1, epochs=2, folds=(0,))
    final = sweep.run_list(finals, RESULTS)[0]
    assert final["status"] == "done", final.get("error")
    test_probs = pd.read_csv(RESULTS / finals[0]["name"] / "test_probs.csv", index_col=0)
    assert len(test_probs) == len(test_csv) and list(test_probs.columns) == LABELS

In [ ]:
with check("summary table builds"):
    table = results.summarize(RESULTS)
    cols = [c for c in ("name", "status", "best_score", "f1_macro_05", "sec_per_epoch", "num_parameters") if c in table]
    display(table[cols])

print("=" * 60)
for name, status in CHECKS.items():
    print(f"{status:5s} {name}")
print("=" * 60)
print("ALL PASS: safe to start the overnight sweep" if all(v == "PASS" for v in CHECKS.values())
      else "NOT READY: send the FAIL output above to Claude")